# Classificazione e transfer learning

Il codice del capitolo [«Classificazione e transfer learning»](https://book.paithon.it/main/VisioneArtificiale/classificazione-transfer.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Classificazione e transfer learning

[Leggi la pagina](https://book.paithon.it/main/VisioneArtificiale/classificazione-transfer.html)


### In pratica, con PyTorch


In [ ]:
import torch
from torch import nn, optim
from torchvision import models

# 1. Rete pre-addestrata su ImageNet, con le sue trasformazioni
pesi = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=pesi)
preprocess = pesi.transforms()   # resize+crop a 224x224, normalizzazione ImageNet

# 2. Feature extraction: si congela tutta la base...
for p in model.parameters():
    p.requires_grad_(False)

# ...e anche i BatchNorm, che altrimenti continuerebbero a cambiare da soli:
# le statistiche sono buffer, non parametri. Da ripetere dopo model.train().
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.eval()

# ...e si sostituisce la testa: dalle 1000 classi ImageNet alle nostre 5
model.fc = nn.Linear(model.fc.in_features, 5)   # nuova, addestrabile

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

In [ ]:
# 3. Si scongela solo l'ultimo blocco della base
for p in model.layer4.parameters():
    p.requires_grad_(True)

optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-5,   # lr basso: cautela
)

## Data augmentation: moltiplicare i dati senza raccoglierli

[Leggi la pagina](https://book.paithon.it/main/VisioneArtificiale/data-augmentation.html)


### In pratica, con torchvision


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

media = (0.485, 0.456, 0.406)   # statistiche di ImageNet: le stesse
dev   = (0.229, 0.224, 0.225)   # usate dalla rete pre-addestrata

# Pipeline di TRAINING: trasformazioni casuali, diverse a ogni epoca
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),  # ritaglio casuale
    transforms.RandomHorizontalFlip(p=0.5),               # specchio nel 50% dei casi
    transforms.ColorJitter(brightness=0.2, contrast=0.2,  # luce, contrasto,
                           saturation=0.2),               # saturazione
    transforms.ToTensor(),
    transforms.Normalize(media, dev),
])

# Pipeline di TEST: deterministica. Niente casualità, mai.
test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(media, dev),
])

train_ds = datasets.ImageFolder("dati/train", transform=train_tf)
test_ds  = datasets.ImageFolder("dati/test",  transform=test_tf)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=32)
```


## Imparare a vedere senza etichette

[Leggi la pagina](https://book.paithon.it/main/VisioneArtificiale/senza-etichette.html)


### Due viste della stessa foto


In [ ]:
import torch
import torch.nn.functional as F

def nt_xent(z, tau=0.5):
    """z: (2N, d). Le prime N righe sono le viste A, le seconde le viste B
    nello stesso ordine: la gemella della riga i e' la riga i+N."""
    n = z.shape[0] // 2
    z = F.normalize(z, dim=1)          # sulla sfera: il prodotto e' un coseno
    sim = (z @ z.t()) / tau            # (2N, 2N) coseni su temperatura
    sim.fill_diagonal_(float("-inf"))  # nessuna vista e' positiva di se stessa
    riga = torch.arange(2 * n, device=z.device)
    bersagli = (riga + n) % (2 * n)    # la gemella della riga i: i+N, oppure i-N
    return F.cross_entropy(sim, bersagli)   # cross-entropy per riga, mediata

### Scegliere le trasformazioni è scrivere il compito


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
N, LATO, RITAGLIO, BIN = 200, 64, 24, 8

# Ogni "immagine" ha una dominante di colore propria (la sua firma cromatica)
# piu' una texture casuale: due ritagli qualunque ne ereditano la dominante.
dominante = rng.uniform(0.2, 0.8, size=(N, 1, 1, 3))
rumore = 0.15 * rng.standard_normal((N, LATO, LATO, 3))
immagini = np.clip(dominante + rumore, 0, 1)

def ritaglia(img):
    i, j = rng.integers(0, LATO - RITAGLIO, size=2)
    return img[i:i + RITAGLIO, j:j + RITAGLIO]

def istogramma(r):
    # 8 bin per canale, concatenati e normalizzati: 24 numeri per ritaglio
    h = [np.histogram(r[..., c], bins=BIN, range=(0, 1))[0] for c in range(3)]
    h = np.concatenate(h).astype(float)
    return h / h.sum()

def accoppia(vista_a, vista_b):
    # per ogni ritaglio di A, il ritaglio di B con l'istogramma piu' vicino
    d = ((vista_a[:, None, :] - vista_b[None, :, :]) ** 2).sum(-1)
    return (d.argmin(axis=1) == np.arange(len(vista_a))).mean()

def colore_casuale(r):
    # jitter di colore: guadagno per canale + luminosita', estratti per ritaglio
    g = rng.uniform(0.6, 1.4, size=3)
    return np.clip(r * g + rng.uniform(-0.2, 0.2), 0, 1)

A = [ritaglia(x) for x in immagini]
B = [ritaglia(x) for x in immagini]

nudi_a = np.array([istogramma(r) for r in A])
nudi_b = np.array([istogramma(r) for r in B])
jit_a = np.array([istogramma(colore_casuale(r)) for r in A])
jit_b = np.array([istogramma(colore_casuale(r)) for r in B])

print("solo ritaglio:          ", round(accoppia(nudi_a, nudi_b), 3))
print("ritaglio + colore:      ", round(accoppia(jit_a, jit_b), 3))
print("livello del caso:       ", round(1 / N, 3))

## Object detection e segmentazione

[Leggi la pagina](https://book.paithon.it/main/VisioneArtificiale/detection-segmentazione.html)


### La famiglia YOLO: le impalcature tolte una alla volta


In [ ]:

# pip install ultralytics; alla prima esecuzione scarica 5 MB di pesi.
from ultralytics import YOLO, ASSETS

modello = YOLO("yolo26n.pt")            # "n" come nano, la taglia più piccola
[esito] = modello(ASSETS / "bus.jpg", verbose=False)
for riquadro in esito.boxes:
    nome = esito.names[int(riquadro.cls)]
    print(f"{nome:10s} confidenza {float(riquadro.conf):.2f}")

## Dove sono le cose: geometria, corrispondenze e profondità

[Leggi la pagina](https://book.paithon.it/main/VisioneArtificiale/geometria-e-profondita.html)


### In pratica: la geometria si può verificare


In [ ]:
import numpy as np

# --- una fotocamera: matrice degli intrinseci ---
f, cx, cy = 700.0, 320.0, 240.0          # focale e centro, in pixel
K = np.array([[f, 0, cx],
              [0, f, cy],
              [0, 0,  1]])

def proietta(K, R, t, P):
    """Porta i punti 3D P (N,3) nel sistema della fotocamera e li proietta."""
    Pc = P @ R.T + t                      # rototraslazione nel sistema camera
    x = Pc @ K.T                          # proiezione prospettica (omogenea)
    return x[:, :2] / x[:, 2:3]           # la divisione per Z: qui si perde la profondità

rng = np.random.default_rng(0)
P = np.column_stack([rng.uniform(-1.5, 1.5, 8),
                     rng.uniform(-1.0, 1.0, 8),
                     rng.uniform( 4.0, 9.0, 8)])   # otto punti davanti alle camere

# --- caso rettificato: seconda camera traslata di B lungo x, stessa orientazione ---
B = 0.30                                   # base, in metri
R1, t1 = np.eye(3), np.zeros(3)
R2, t2 = np.eye(3), np.array([-B, 0.0, 0.0])

u1 = proietta(K, R1, t1, P)
u2 = proietta(K, R2, t2, P)

disparita = u1[:, 0] - u2[:, 0]
Z_stimata = f * B / disparita

print("profondità vera   :", np.round(P[:, 2], 3))
print("profondità stimata:", np.round(Z_stimata, 3))
print("errore massimo    :", np.abs(Z_stimata - P[:, 2]).max())
print("righe uguali (rettificato):", np.allclose(u1[:, 1], u2[:, 1]))

In [ ]:
ang = np.deg2rad(8.0)
Ry = np.array([[ np.cos(ang), 0, np.sin(ang)],   # rotazione attorno all'asse y
               [ 0,           1, 0          ],   # (la verticale): la seconda riga
               [-np.sin(ang), 0, np.cos(ang)]])  # e colonna restano identiche
t3 = np.array([-B, 0.02, 0.0])
u3 = proietta(K, Ry, t3, P)

def antisimmetrica(v):
    """La matrice che realizza il prodotto vettoriale: [v]_x @ w == np.cross(v, w)."""
    return np.array([[0, -v[2], v[1]],
                     [v[2], 0, -v[0]],
                     [-v[1], v[0], 0]])

F = np.linalg.inv(K).T @ antisimmetrica(t3) @ Ry @ np.linalg.inv(K)

def omogenee(u):
    return np.column_stack([u, np.ones(len(u))])

residuo = np.einsum('ij,jk,ik->i', omogenee(u3), F, omogenee(u1))
print("residuo epipolare :", np.abs(residuo).max())

## Scene che si addestrano: NeRF e splatting

[Leggi la pagina](https://book.paithon.it/main/VisioneArtificiale/rendering-neurale.html)


### In pratica: la composizione lungo un raggio


In [ ]:
import numpy as np

def rendi_raggio(sigma, colori, delta):
    """Composizione volumetrica lungo un raggio.
    sigma: densità per campione; colori: (N,3); delta: passo fra i campioni."""
    alpha = 1.0 - np.exp(-sigma * delta)                 # quanto ogni campione occlude
    trasmittanza = np.cumprod(np.concatenate([[1.0], 1.0 - alpha[:-1]]))
    pesi = trasmittanza * alpha                          # quanto ogni campione conta
    return (pesi[:, None] * colori).sum(axis=0), pesi

N, lunghezza = 60, 6.0
delta = lunghezza / N                                    # 10 cm fra un campione e l'altro
t = np.arange(N) * delta                                 # 0.0, 0.1, ... 5.9 metri

# vuoto, e a quattro metri una superficie opaca
sigma = np.where(np.isclose(t, 4.0), 60.0, 0.0)
colori = np.tile(np.array([0.71, 0.33, 0.17]), (N, 1))   # terracotta

C, pesi = rendi_raggio(sigma, colori, delta)
print("colore reso       :", np.round(C, 3))
print("massa dei pesi    :", round(float(pesi.sum()), 4))
print("profondità attesa :", round(float((pesi * t).sum()), 3), "m")

# la stessa scena riempita di nebbia: nessuna superficie, i pesi si spalmano
C2, pesi2 = rendi_raggio(np.full(N, 0.45), colori, delta)
print("massa con la nebbia:", round(float(pesi2.sum()), 4),
      "| il picco dei pesi vale", round(float(pesi2.max()), 4),
      "contro", round(float(pesi.max()), 4), "della superficie")

## Neural style transfer: la tua foto dipinta da van Gogh

[Leggi la pagina](https://book.paithon.it/main/VisioneArtificiale/style-transfer.html)


### In pratica, con PyTorch


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch
from torch import nn, optim
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. VGG-19 pre-addestrata: solo la parte convoluzionale, congelata
vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
vgg = vgg.features.to(device).eval()
for p in vgg.parameters():
    p.requires_grad_(False)

STRATI_STILE = [0, 5, 10, 19, 28]   # conv1_1 ... conv5_1
STRATO_CONTENUTO = 21               # conv4_2

def attivazioni(x):
    stile, contenuto = [], None
    for i, strato in enumerate(vgg):
        x = strato(x)
        if i in STRATI_STILE:
            stile.append(x)
        elif i == STRATO_CONTENUTO:
            contenuto = x
        if i == STRATI_STILE[-1]:
            break                    # oltre conv5_1 non serve
    return stile, contenuto

def gram(f):
    _, c, h, w = f.shape             # f: (1, c, h, w)
    F = f.view(c, h * w)
    return F @ F.T / (c * h * w)     # Gram (c, c), normalizzata a modo nostro:
                                     # non e' la 1/(4 N^2 M^2) del paper, quindi
                                     # il beta qui sotto non e' quello del paper

# img_contenuto, img_stile: tensori (1, 3, H, W) già ridimensionati
# e normalizzati con media e deviazione standard di ImageNet
with torch.no_grad():
    stile_rif, _ = attivazioni(img_stile)
    _, contenuto_rif = attivazioni(img_contenuto)
    gram_rif = [gram(f) for f in stile_rif]

# 2. Si ottimizza l'IMMAGINE: parte dalla foto, il gradiente scende sui pixel
img = img_contenuto.clone().requires_grad_(True)
opt = optim.Adam([img], lr=0.02)
alpha, beta = 1.0, 1e5           # taglia solidale con la gram() qui sopra:
                                 # il 1000 della sezione precedente vale per
                                 # la normalizzazione del paper, non per questa

for passo in range(300):
    opt.zero_grad()
    stile_gen, contenuto_gen = attivazioni(img)
    l_contenuto = nn.functional.mse_loss(contenuto_gen, contenuto_rif)
    l_stile = sum(nn.functional.mse_loss(gram(f), g)
                  for f, g in zip(stile_gen, gram_rif))
    loss = alpha * l_contenuto + beta * l_stile
    loss.backward()
    opt.step()
```
